In [1]:
import tensorflow as tf
import os
import numpy as np
import cv2
import shutil

from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
BASE_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/dental_xray"
PATCH_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/dental_patches"

IMG_SIZE = 224
PATCH_SIZE = 224
STRIDE = 112

if os.path.exists(PATCH_DIR):
    shutil.rmtree(PATCH_DIR)

for split in ["train", "val", "test"]:
    for cls in ["CAVITY", "NORMAL"]:
        os.makedirs(os.path.join(PATCH_DIR, split, cls), exist_ok=True)

In [3]:
def extract_patches(img, patch_size=224, stride=112):
    patches = []
    h, w, _ = img.shape

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = img[y:y+patch_size, x:x+patch_size]
            patches.append((patch, x, y))

    return patches

In [4]:
def is_valid_patch(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    return np.std(gray) > 15   # remove blank/low-detail patches

In [5]:
def get_top_patches(patches, top_k=0.4):
    scored = []

    for patch, x, y in patches:
        score = np.std(cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY))
        scored.append((score, patch, x, y))

    scored.sort(reverse=True)

    k = int(len(scored) * top_k)

    return [(p, x, y) for (_, p, x, y) in scored[:k]]

In [6]:
def process_dataset(split):
    input_path = os.path.join(BASE_DIR, split)

    for cls in ["CAVITY", "NORMAL"]:
        class_path = os.path.join(input_path, cls)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            img = cv2.resize(img, (512, 512))

            patches = extract_patches(img)

            # 🔥 Step 1: remove useless patches
            patches = [p for p in patches if is_valid_patch(p[0])]

            # 🔥 Step 2: reduce noise for cavity
            if cls == "CAVITY":
                patches = get_top_patches(patches, top_k=0.4)

            for i, (patch, x, y) in enumerate(patches):
                save_path = os.path.join(
                    PATCH_DIR, split, cls,
                    f"{img_name}_{i}.jpg"
                )
                cv2.imwrite(save_path, patch)


for split in ["train", "val", "test"]:
    process_dataset(split)

print("✅ Patch dataset created (filtered + optimized)")

✅ Patch dataset created (filtered + optimized)


In [7]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
).flow_from_directory(
    os.path.join(PATCH_DIR, "train"),
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    os.path.join(PATCH_DIR, "val"),
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    os.path.join(PATCH_DIR, "test"),
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 1008 images belonging to 2 classes.
Found 213 images belonging to 2 classes.
Found 216 images belonging to 2 classes.


In [8]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),   # 🔥 lower LR
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "backend/saved_models/dental_patch_model_final.keras",
        save_best_only=True
    )
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6657 - loss: 0.7174 - val_accuracy: 0.7418 - val_loss: 0.6120
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 26s 831ms/step - accuracy: 0.7024 - loss: 0.6112 - val_accuracy: 0.7136 - val_loss: 0.6076
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 31s 965ms/step - accuracy: 0.7312 - loss: 0.5708 - val_accuracy: 0.7324 - val_loss: 0.5876
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.7321 - loss: 0.5508 - val_accuracy: 0.7324 - val_loss: 0.5802
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.7460 - loss: 0.5281 - val_accuracy: 0.6901 - val_loss: 0.5900
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.7569 - loss: 0.5017 - val_accuracy: 0.6948 - val_loss: 0.5952
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.7639 - loss: 0.5095 - val_accuracy: 0.6901 - val_loss: 0.5755
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.7698 - loss: 0.4862 - val_accuracy: 0.6995 - val

In [10]:
loss, acc = model.evaluate(test_gen)
print("✅ Final Patch Model Accuracy:", acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 660ms/step - accuracy: 0.7361 - loss: 0.5659
✅ Final Patch Model Accuracy: 0.7361111044883728


In [11]:
def predict_image(image_path, model, threshold=0.6):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (512,512))

    patches = extract_patches(img)

    results = []

    for patch, x, y in patches:
        if not is_valid_patch(patch):
            continue

        patch_resized = cv2.resize(patch, (224,224))
        patch_resized = preprocess_input(patch_resized)
        patch_resized = np.expand_dims(patch_resized, axis=0)

        pred = model.predict(patch_resized, verbose=0)[0][0]

        if pred > threshold:
            results.append((x, y, pred))

    return results

In [12]:
import matplotlib.pyplot as plt

def show_predictions(image_path, model):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (512,512))

    results = predict_image(image_path, model)

    for (x, y, conf) in results:
        cv2.rectangle(img, (x,y), (x+224, y+224), (0,0,255), 2)

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Cavities detected: {len(results)}")
    plt.axis("off")
    plt.show()

In [15]:
loss, acc = model.evaluate(test_gen)
print("✅ Final Patch Model Accuracy:", acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 626ms/step - accuracy: 0.7500 - loss: 0.5566
✅ Final Patch Model Accuracy: 0.75


In [ ]:
FINAL_MODEL_PATH = "backend/saved_models/dental_final.keras"

model.save(FINAL_MODEL_PATH)

print("✅ Final model saved at:", FINAL_MODEL_PATH)

✅ Final model saved at: backend/saved_models/dental_final.keras


In [14]:
model = tf.keras.models.load_model("backend/saved_models/dental_final.keras")